In [3]:
import tensorflow as tf


(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

print(f"\nLoaded CIFAR-10 dataset:")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")


Loaded CIFAR-10 dataset:
X_train shape: (50000, 32, 32, 3), y_train shape: (50000, 1)
X_test shape: (10000, 32, 32, 3), y_test shape: (10000, 1)


In [5]:
#using VGG16

from tensorflow.keras import layers, models
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

vgg_16_base_model = VGG16(
    input_shape = (224,224,3),
    weights = "imagenet",
    include_top = False
)


vgg_16_base_model.trainable = False     #freezing the base


vgg_16_model = models.Sequential([
    layers.Input(shape=(32,32,3)), #input shape now matches CIFAR-10 data
    layers.Resizing(224, 224),      #resize images to 224x224 for vgg16
    layers.Lambda(preprocess_input),
    vgg_16_base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10,activation="softmax")

])


vgg_16_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = ["accuracy"]
)


vgg16_history = vgg_16_model.fit(
    X_train,
    y_train,
    batch_size = 64,
    validation_data = (X_test, y_test),
    epochs = 3
)

Epoch 1/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 349s 444ms/step - accuracy: 0.7396 - loss: 0.8075 - val_accuracy: 0.8226 - val_loss: 0.5289
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 343s 439ms/step - accuracy: 0.8447 - loss: 0.4601 - val_accuracy: 0.8390 - val_loss: 0.4828
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 342s 438ms/step - accuracy: 0.8582 - loss: 0.4156 - val_accuracy: 0.8444 - val_loss: 0.4648


The model rapidly converges, jumping from baseline guessing straight to a strong 73.96% training accuracy and 82.26% validation accuracy in the very first epoch. By the third epoch, training accuracy reaches 85.82% while validation accuracy tops out at 84.44%.

Because validation accuracy closely tracks training accuracy without a steep divergence, the frozen VGG16 features generalize cleanly to CIFAR-10 without severe overfitting.



Setting trainable = False freezes VGG16's pre-trained ImageNet weights to act strictly as a feature extractor.